# HEC Match — TabICL MITIGÉ (sans proxies raciaux)

Ce notebook ré-entraîne TabICL en retirant les **4 variables proxy** identifiées par FPDP qui causaient le biais racial.

**Variables retirées :**
- `attr3_1_B` — auto-évaluation d'attractivité du candidat B
- `career_c_B_12` — code carrière spécifique
- `go_out_B` — fréquence de sorties
- `intel3_1_B` — auto-évaluation d'intelligence

### ⚠️ Avant de lancer
1. **Runtime GPU obligatoire** : `Modifier > Paramètres du notebook > GPU (T4)`
2. **Uploader ces 3 fichiers** (icône dossier à gauche) :
   - `data/clean.parquet` → `clean.parquet`
   - `data/features.json` → `features.json`
   - `data/split.json` → `split.json`

In [ ]:
!pip install -q tabicl

In [ ]:
import importlib.metadata
import json
import platform
import numpy as np
import pandas as pd
import joblib
import sklearn
import torch
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from tabicl import TabICLClassifier

SEED = 42
PROXY_FEATURES_TO_DROP = [
    "attr3_1_B",
    "career_c_B_12",
    "go_out_B",
    "intel3_1_B",
]

print("platform:", platform.platform(), "|", platform.machine())
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("sklearn:", sklearn.__version__, "| tabicl:", importlib.metadata.version("tabicl"))
assert torch.cuda.is_available(), "⛔ Pas de GPU — Modifier > Paramètres du notebook > GPU, puis relancer."

## 1. Chargement des données

In [ ]:
data = pd.read_parquet("clean.parquet")
feature_dict = json.load(open("features.json"))
split = json.load(open("split.json"))

TARGET = feature_dict["target"]
FEATURES_BASE = feature_dict["features"]  # 158 features (comme XGBoost)
FEATURES_MIT = [f for f in FEATURES_BASE if f not in PROXY_FEATURES_TO_DROP]  # 154 features

dropped = [f for f in PROXY_FEATURES_TO_DROP if f in FEATURES_BASE]
missing = [f for f in PROXY_FEATURES_TO_DROP if f not in FEATURES_BASE]
print(f"Features base : {len(FEATURES_BASE)} | Features mitigées : {len(FEATURES_MIT)}")
print(f"Proxies retirés ({len(dropped)}) : {dropped}")
if missing:
    print(f"[ATTENTION] Proxies déjà absents de features.json : {missing}")

## 2. Split train / test (identique au pipeline principal)

In [ ]:
train_waves, test_waves = set(split["train_waves"]), set(split["test_waves"])
assert not (train_waves & test_waves), "waves communes train/test"

train = data[data["wave"].isin(train_waves)].reset_index(drop=True)
test  = data[data["wave"].isin(test_waves)].reset_index(drop=True)

X_tr, y_tr = train[FEATURES_MIT], train[TARGET]
X_te, y_te = test[FEATURES_MIT],  test[TARGET]
print(f"train : {X_tr.shape} | test : {X_te.shape}")

## 3. Entraînement TabICL mitigé (train complet)

In [ ]:
model = make_pipeline(
    SimpleImputer(strategy="median"),
    TabICLClassifier(random_state=SEED, device="cuda"),
)
model.fit(X_tr, y_tr)
print("✅ fit OK")

proba_test = model.predict_proba(X_te)[:, 1]
auc_test = roc_auc_score(y_te, proba_test)
print(f"AUC test (split unique) : {auc_test:.3f}  (référence base non mitigé : 0.632)")

## 4. GroupKFold 5 folds (validation croisée par wave)

In [ ]:
X_all, y_all, groups_all = data[FEATURES_MIT], data[TARGET], data["wave"]
gkf = GroupKFold(n_splits=5)
aucs = []
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_all, y_all, groups=groups_all), start=1):
    m = make_pipeline(
        SimpleImputer(strategy="median"),
        TabICLClassifier(random_state=SEED, device="cuda"),
    )
    m.fit(X_all.iloc[tr_idx], y_all.iloc[tr_idx])
    auc_f = roc_auc_score(y_all.iloc[te_idx], m.predict_proba(X_all.iloc[te_idx])[:, 1])
    aucs.append(auc_f)
    print(f"Fold {fold} : AUC = {auc_f:.3f}")

aucs = np.array(aucs)
print(f"\nGroupKFold : {aucs.mean():.3f} ± {aucs.std():.3f}  (référence base : 0.591 ± 0.038)")

## 5. Export — modèle + prédictions sur dataset complet

In [ ]:
# Modèle (CUDA-only — ne se charge pas hors Colab/GPU)
joblib.dump(model, "tabicl_mitigated.joblib")
print("Modèle sauvegardé : tabicl_mitigated.joblib")

# Prédictions sur le dataset complet (pour les scripts de fairness en local)
proba_full = model.predict_proba(data[FEATURES_MIT])[:, 1]
preds_df = pd.DataFrame({
    "iid": data["iid"],
    "pid": data["pid"],
    "wave": data["wave"],
    "tabicl_mit_proba": proba_full,
})
preds_df.to_parquet("tabicl_mitigated_predictions.parquet", index=False)
print(f"Prédictions exportées : {len(preds_df)} lignes -> tabicl_mitigated_predictions.parquet")

## 6. Téléchargement des fichiers produits

Après le téléchargement, placer les fichiers dans le repo :
- `tabicl_mitigated.joblib` → `models/`
- `tabicl_mitigated_predictions.parquet` → `data/`

In [ ]:
from google.colab import files
files.download("tabicl_mitigated.joblib")
files.download("tabicl_mitigated_predictions.parquet")
print("✅ Téléchargement lancé.")